In [1]:
import torch
import torchaudio

print(torch.__version__)
print(torchaudio.__version__)

2.8.0+cu128
2.8.0+cu128


In [2]:
from fairseq2 import gang
gang._thread_local.current_gangs = []

In [3]:
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
pipeline = ASRInferencePipeline(model_card= 'omniASR_LLM_300M')

Output()

In [6]:
import subprocess

def encode_to_wav(audio):
    encoded_audio = subprocess.run(
        ['ffmpeg', '-i', audio, '-f', 'wav', 'pipe:1'],
        check = True,
        capture_output= True
    )
    return encoded_audio.stdout #stdout is the actual file

audio = 'sw-test-speech-1.m4a'
encoded_audio = encode_to_wav(audio)
type(encoded_audio)

bytes

In [22]:
import io, soundfile as sf
#obtained bytes go to a memory like object

audio = io.BytesIO(encoded_audio)
audio.seek(0)

waveform, sr = sf.read(audio)
waveform = torch.from_numpy(waveform).float()

In [26]:
print(f'Sample rate: {sr}')

Sample rate: 48000


In [27]:
import tempfile

with tempfile.NamedTemporaryFile(suffix='.wav', delete= False) as tmp:
    sf.write(tmp.name, waveform.numpy(), sr)
    transcript = pipeline.transcribe([tmp.name], batch_size= 1)

print(transcript)

['mimi anaitwa nathan una akiukweli napenda walisamaki yaani akiongelea walisamaki naongelea ni rosten ni ule wale ambao yaani samaki wake unakuwa unaoroja uroja yaani unakuwa mtaa tunaopenda']
